In [5]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import pandas as pd
import torch

import faiss
import numpy as np

from langchain_core.documents import Document

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from rouge_score import rouge_scorer
from bert_score import score as bertscore

from tqdm.auto import tqdm

from huggingface_hub import whoami
from sklearn.model_selection import train_test_split

import gc
import torch

In [2]:
!pip install faiss-gpu
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 14.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.1 MB/s eta 0:00:00:00:0100:01


In [3]:
!pip install rouge_score
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.3 MB/s eta 0:00:00


In [4]:
#!pip install -U "bitsandbytes>=0.46.1"
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.6 MB/s eta 0:00:00:00:0100:01


**LOAD FINAL CORPUS + EMBEDDING MODELS**

In [6]:

# ---------- Configuration ----------
CORPUS_ID = "vab46/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data_final"

FT_EMBED_ID = "vab46/nomic-embed-text-v1.5_Clinical-Trials_Matryoshka_final"

BASE_EMBED_ID = "nomic-ai/nomic-embed-text-v1.5"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ---------- Load final embedding dataset ----------
ds = load_dataset(CORPUS_ID)
split = "train" if "train" in ds else list(ds.keys())[0]

pairs_df = ds[split].to_pandas()

print(f"Anchor-positive pairs loaded: {len(pairs_df):,}")


# ---------- Validate expected schema ----------
EXPECTED_COLUMNS = [
    "anchor_id",
    "nctId",
    "chunk_type",
    "block_no",
    "document_id",
    "anchor_type",
    "anchor",
    "positive",
    "chunk_char_length",
    "status",
]

missing = [c for c in EXPECTED_COLUMNS if c not in pairs_df.columns]

if missing:
    raise ValueError(f"Missing expected columns: {missing}")


# ---------- Prepare unique retrieval corpus ----------
# Same positive/document chunk can occur for 4 different anchors.
# Keep ONE corpus record per document/chunk identity.

pairs_df = pairs_df[
    pairs_df["positive"].notna()
    & pairs_df["positive"].astype(str).str.strip().ne("")
].copy()

CORPUS_KEY = [
    "nctId",
    "document_id",
    "chunk_type",
    "block_no",
]

corpus_df = (
    pairs_df
    .drop_duplicates(subset=CORPUS_KEY)
    .reset_index(drop=True)
)

corpus_texts = corpus_df["positive"].astype(str).tolist()

print(f"Unique retrieval chunks: {len(corpus_df):,}")
print(f"Embedding device: {DEVICE}")


# Load FT & base embedding model ----------
ft_embed_model = SentenceTransformer(
    FT_EMBED_ID,
    device=DEVICE,
    trust_remote_code=True,
)


base_embed_model = SentenceTransformer(
    BASE_EMBED_ID,
    device=DEVICE,
    trust_remote_code=True,
)


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.03M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7866 [00:00<?, ? examples/s]

Anchor-positive pairs loaded: 7,866
Unique retrieval chunks: 2,000
Embedding device: cuda


modules.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

...

<All keys matched successfully>


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [7]:
EMBED_DIM =768

In [8]:
# Sanity-check both models ----------
test_text = ["clinical trial eligibility"]

ft_test = ft_embed_model.encode(
    test_text,
    convert_to_numpy=True,
    normalize_embeddings=True,
    truncate_dim=EMBED_DIM #added only to truncate dim
)

base_test = base_embed_model.encode(
    test_text,
    convert_to_numpy=True,
    normalize_embeddings=True,
    truncate_dim=EMBED_DIM #added only to truncate dim
)

assert ft_test.shape[1] == EMBED_DIM, (
    f"Unexpected FT embedding dimension: {ft_test.shape[1]}"
)

assert base_test.shape[1] == EMBED_DIM, (
    f"Unexpected Base embedding dimension: {base_test.shape[1]}"
)

print(f"FT embedding dimension   : {ft_test.shape[1]}")
print(f"Base embedding dimension : {base_test.shape[1]}")

print("\nBLOCK 1 PASSED")

FT embedding dimension   : 768
Base embedding dimension : 768

BLOCK 1 PASSED


**BUILD BASE + FT FAISS INDEXES**

In [9]:
import faiss
import numpy as np


# ---------- Encode the same corpus with both models ----------
# normalize_embeddings=True → cosine similarity via inner product

ft_embeddings = ft_embed_model.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    truncate_dim= EMBED_DIM #added only to truncate dim
)

base_embeddings = base_embed_model.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    truncate_dim= EMBED_DIM #added only to truncate dim
)


# ---------- Convert to FAISS-compatible dtype ----------
ft_embeddings = np.asarray(ft_embeddings, dtype="float32")
base_embeddings = np.asarray(base_embeddings, dtype="float32")


# ---------- Sanity checks ----------
assert ft_embeddings.shape[0] == len(corpus_df)
assert base_embeddings.shape[0] == len(corpus_df)

assert ft_embeddings.shape[1] == EMBED_DIM
assert base_embeddings.shape[1] == EMBED_DIM


# ---------- Build FAISS indexes ----------
# Inner Product on normalized vectors = cosine similarity

#ft_faiss_index = faiss.IndexFlatIP(768)
#base_faiss_index = faiss.IndexFlatIP(768)

ft_faiss_index = faiss.IndexFlatIP(EMBED_DIM)
base_faiss_index = faiss.IndexFlatIP(EMBED_DIM)


ft_faiss_index.add(ft_embeddings)
base_faiss_index.add(base_embeddings)


# ---------- Final validation ----------
assert ft_faiss_index.ntotal == len(corpus_df)
assert base_faiss_index.ntotal == len(corpus_df)

print(f"FT FAISS vectors   : {ft_faiss_index.ntotal:,}")
print(f"Base FAISS vectors : {base_faiss_index.ntotal:,}")
print(f"Vector dimension   : {ft_faiss_index.d}")
print("\nBLOCK 2 PASSED")

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

FT FAISS vectors   : 2,000
Base FAISS vectors : 2,000
Vector dimension   : 768

BLOCK 2 PASSED


**QUERY ENCODING + FAISS RETRIEVAL+LANG CHIAN INTEGRATION+SANITY CHECKS**

In [10]:
# query encoding + faisss retrieval---------------------------------------

def retrieve_chunks(
    query,
    embedding_model,
    faiss_index,
    corpus_df,
    top_k=5,
):
    """
    Encode a user question and retrieve top-k corpus chunks.

    Same encoder is used for query and corpus within each pipeline.
    Embeddings are normalized, so FAISS Inner Product = cosine similarity.
    """

    # Query embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
        truncate_dim= EMBED_DIM #added only to truncate dim
    ).astype("float32")

    # FAISS retrieval
    scores, indices = faiss_index.search(
        query_embedding,
        top_k,
    )

    # Map FAISS indices back to corpus records
    results = corpus_df.iloc[indices[0]].copy().reset_index(drop=True)
    results["retrieval_score"] = scores[0]

    return results


# langchain integration wrapper---------------------------

def langchain_retrieve(
    query,
    embedding_model,
    faiss_index,
    corpus_df,
    top_k=5,
):
    """
    LangChain-compatible retrieval wrapper.

    Uses the existing retrieve_chunks() function from Block 3
    and converts retrieved corpus rows into LangChain Documents.
    """

    results = retrieve_chunks(
        query=query,
        embedding_model=embedding_model,
        faiss_index=faiss_index,
        corpus_df=corpus_df,
        top_k=top_k,
    )

    documents = []

    for _, row in results.iterrows():
        documents.append(
            Document(
                page_content=row["positive"],
                metadata={
                    "nctId": row["nctId"],
                    "document_id": row["document_id"],
                    "chunk_type": row["chunk_type"],
                    "block_no": row["block_no"],
                    "retrieval_score": float(row["retrieval_score"]),
                },
            )
        )

    return documents

print("LangChain retrieval integration ready.")

LangChain retrieval integration ready.


In [11]:
#sanity checks-----------------------
test_query = pairs_df.iloc[3]["anchor"]

ft_retrieval = langchain_retrieve(
    test_query,
    ft_embed_model,
    ft_faiss_index,
    corpus_df,
    top_k=5,
)

base_retrieval = langchain_retrieve(
    test_query,
    base_embed_model,
    base_faiss_index,
    corpus_df,
    top_k=5,
)

print("Query:", test_query)
print("\nFT top result:")
print(ft_retrieval[0].page_content[:1000])
print(ft_retrieval[0].metadata)

print("\nBase top result:")
print(base_retrieval[0].page_content[:1000])
print(base_retrieval[0].metadata)

Query: I had a small stroke and my doctor said I should be more active. Can I use a smartwatch to help me walk more and reduce my risk of another stroke?

FT top result:
TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Guided Secondary Prevention After Stroke Randomized Trial of Nurse-led Program With Active vs Passive Smartwatch in Minor Stroke. A Randomized Controlled Trial Evaluating a Nurse-led Secondary Prevention and Physical Activity Program Supported by Either an Active Smartwatch (Structured Feedback) or Passive Smartwatch in Patients With Minor Stroke.
SUMMARY: After a first stroke or transient ischemic attack (TIA), the risk of recurrence is high in the weeks and months following the initial event. There are several modifiable risk factors that can reduce this risk, such as blood pressure, diet, physical activity, and smoking. Many stroke patients (NIHSS \< 5) have a low daily step count during the early recovery period, despite a good functional prognosis.
Active smartwatches pr

**authenticate secret raed/write HF tokens**

In [12]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

from huggingface_hub import login

login(token=HF_TOKEN)

print(whoami())

{'type': 'user', 'id': '6a5e70dfab67e71b761fb3f3', 'name': 'vab46', 'fullname': 'Vaibhav Raj', 'isPro': False, 'avatarUrl': '/avatars/d689d000ad2f3233cb26a6ca99618cf1.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'colab-read-write', 'role': 'fineGrained', 'createdAt': '2026-08-16T12:11:43.414Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write'], 'scoped': [{'entity': {'_id': '6a5e70dfab67e71b761fb3f3', 'type': 'user', 'name': 'vab46'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.write', 'discussion.write']}]}}}}


**loading Base and LORA LLM generator**

In [16]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
#LORA_ID = "vab46/llama-3.1-8b-instruct-lora-clinical_iter1"#iter1
LORA_ID = "vab46/llama-3.1-8b-instruct-lora-clinical_iter2_epoch2"#iter2-epoch2

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

#base mode loading-------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

#freating ft model from base-----------------
ft1_model = PeftModel.from_pretrained(
    base_model,
    LORA_ID,
)

base_model.eval()# optional as base model by deafult in eval mode
ft1_model.eval()

print("Base model loaded.")
print("LoRA adapter loaded.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Base model loaded.
LoRA adapter loaded.


**RAG PROMPT + GENERATION**

In [17]:
#iter=1
iter =2

In [18]:
#Rag-prompt--------------------------------------------
if(iter ==1):
    RAG_SYSTEM_PROMPT = """You are a clinical-trial question answering assistant.
    
    Answer the user's QUESTION using the provided CONTEXT.
    
    - Answer directly and naturally.
    - Base the answer on the CONTEXT and do not introduce unsupported trial-specific facts.
    - Synthesize relevant information rather than simply copying the context.
    - Include important numbers, thresholds, dates, age ranges, conditions, interventions, and eligibility criteria when relevant.
    - For eligibility questions, explain whether the information given satisfies the relevant criteria and mention other important criteria when relevant.
    - Do not include irrelevant trial information.
    - If the CONTEXT does not provide enough information, say so clearly.
    - Preserve trial-specific details accurately.
    - Answer concisely, factually, and sufficiently completely.
    """
else:

    RAG_SYSTEM_PROMPT = """You are a clinical-trial question answering assistant.

    Answer the user's QUESTION using the provided CONTEXT.

    - Answer directly and naturally.
    - Base the answer on the CONTEXT and do not introduce unsupported trial-specific facts.
    - Synthesize relevant information rather than simply copying the context.
    - Include important numbers, thresholds, dates, age ranges, conditions, interventions, and eligibility criteria when relevant.
    - For eligibility questions, explain whether the information given satisfies the relevant criteria and mention other important criteria when relevant.
    - Do not include irrelevant trial information.
    - If the CONTEXT does not provide enough information, say so clearly.
    - Preserve trial-specific details accurately.
    - Answer concisely, factually, and sufficiently completely.
    - Don't limit answers to single word(1.eligiblity/qualification questions with a 'True/False'; 2. age/temporal questions with just a 'number') or blank(in case no answer avalilable in a given context).Rather 
      briefly explain the answer in with supported criteria/reasons(from context).
    - For eligiblity/qualification questions, preserve ALL relevant eligibility/inclusion criteria supported by the CONTEXT. If distinct criteria/reasons (generally >3-4) are relevant and long string based continous 
      sentence affects context, distinguish between criteria and additional criteria if supported by context .State the additional criteria explicitly(json). Else stay with string based continous format in other cases.
    """

#response-generation---------------------------------------

def generate_answer(model, question, retrieved_docs, max_new_tokens=256):
    context = "\n\n".join(
        f"CONTEXT {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(retrieved_docs)
    )

    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"QUESTION:\n{question}\n\n{context}",
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    #print(inputs)#test_statement
    #print(inputs['input_ids'].shape)#test_sattement

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    #print(output)#test_statement
    #print(output.shape)#test_statemnt

    generated_tokens = output[0][inputs['input_ids'].shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()



**End-to-end RAG inference(using both ft and base generator LLM) on a sample**

In [19]:
# end-to-end refernce fn----------------------------------

def rag_answer(
    question,
    mode="ft",
    top_k=5,
    max_new_tokens=256,
):
    if mode == "ft":
        embedding_model = ft_embed_model
        faiss_index = ft_faiss_index
        generator_model = ft1_model
    elif mode == "base":
        embedding_model = base_embed_model
        faiss_index = base_faiss_index
        generator_model = base_model
    else:
        raise ValueError("mode must be 'ft' or 'base'")

    retrieved_docs = langchain_retrieve(
        question,
        embedding_model,
        faiss_index,
        corpus_df,
        top_k=top_k,
    )

    answer = generate_answer(
        generator_model,
        question,
        retrieved_docs,
        max_new_tokens=max_new_tokens,
    )

    return {
        "question": question,
        "answer": answer,
        "retrieved_docs": retrieved_docs,
    }

# base vs ft end to end generation sanity test----------

test_question = pairs_df.iloc[2]["anchor"]

base_result = rag_answer(
    test_question,
    mode="base",
    top_k=4,
)

ft_result = rag_answer(
    test_question,
    mode="ft",
    top_k=4,
)

print("QUESTION:")
print(test_question)

print("\nBASE ANSWER:")
print(base_result["answer"])

print("\nFT + LoRA ANSWER:")
print(ft_result["answer"])

print("\nBASE RETRIEVED NCT IDs:")
print([
    (doc.metadata["nctId"], round(doc.metadata["retrieval_score"], 4))
    for doc in base_result["retrieved_docs"]
])

print("\nFT RETRIEVED NCT IDs:")
print([
    (doc.metadata["nctId"], round(doc.metadata["retrieval_score"], 4))
    for doc in ft_result["retrieved_docs"]
])

QUESTION:
Is there an age limit for patients to participate in this trial, and do they need to have WiFi access at home?

BASE ANSWER:
There is no age limit for patients to participate in this trial. However, the study focuses on older adults who live alone and have memory concerns or mild cognitive decline. Participants must also have access to a computer or mobile device and internet service.

FT + LoRA ANSWER:
The provided context does not contain enough information to answer whether there is an age limit for patients to participate in this trial, or if they need to have WiFi access at home.

BASE RETRIEVED NCT IDs:
[('NCT07659548', 0.7045), ('NCT07659379', 0.7014), ('NCT07634406', 0.6887), ('NCT07705945', 0.6873)]

FT RETRIEVED NCT IDs:
[('NCT07640698', 0.4616), ('NCT07706114', 0.4382), ('NCT07640360', 0.437), ('NCT07634406', 0.4107)]


In [21]:
# ============================================================
# RETRIEVAL INSPECTION
# ============================================================

for name, result in [("BASE", base_result), ("FT + LoRA", ft_result)]:
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")

    for i, doc in enumerate(result["retrieved_docs"], 1):
        print(
            f"\n[{i}] NCT: {doc.metadata['nctId']} | "
            f"score: {doc.metadata['retrieval_score']:.4f}"
        )
        print(doc.page_content[:300].replace("\n", " "))


BASE

[1] NCT: NCT07659548 | score: 0.7045
TITLE: Technological Innovation to Support Medication Safety and Optimization in Older Adults Living Alone With Cognitive Decline SUMMARY: This clinical trial is studying an online tool that may help older adults safely manage their medications at home. The study focuses on older adults who live alo

[2] NCT: NCT07659379 | score: 0.7014
TITLE: Extension Trial to Study the Long-term Safety and Efficacy in Participants With Advanced Tumors Who Are Currently on Treatment or in Follow-up in Gotistobart Trials SUMMARY: This LTE study will examine the long-term overall survival and safety of gotistobart in patients with advanced solid tu

[3] NCT: NCT07634406 | score: 0.6887
TITLE: The Effect of a Nurse-Led Telehealth-Supported Multicomponent Frailty Management Program on Frailty and Functional Status in Older Adults Living at Home: A Randomized Controlled Trial SUMMARY: Frailty is an important geriatric syndrome associated with reduced physiologic

### **Evaluation**

**Initilalize ref_answer dataframe(anchor. context/positive, refernce_answer/grounTruth)evaluated RAG set(question, retreieval docs, retrieval score, generated answer) & append refernce_answer Dataframe**

In [22]:
#HF_DATASET_ID = "vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA_ft"#--->iter1
HF_DATASET_ID = "vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA-junk_handled_ft"#--->iter2

# Load final cleaned dataset
ref_ans_ds = load_dataset(HF_DATASET_ID)

# Use train split (assuming the uploaded dataset has the default train split)
ref_ans_df = ref_ans_ds["train"].to_pandas()

print("Rows:", len(ref_ans_df))
print("\nColumns:")
print(ref_ans_df.columns.tolist())

print("\nReference-answer types:")
print(ref_ans_df["reference_answer"].apply(type).value_counts())

print("\nStatus:")
print(ref_ans_df["status"].value_counts())

print("\nUnique documents:")
print(ref_ans_df["document_id"].nunique())

print("\nRows per document — summary:")
print(ref_ans_df.groupby("document_id").size().describe())

print("\nSample:")
display(
    ref_ans_df[
        ["anchor_id", "nctId", "document_id",
         "anchor_type", "anchor", "positive", "reference_answer"]
    ].head(3)
)



README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7866 [00:00<?, ? examples/s]

Rows: 7866

Columns:
['Unnamed: 0', 'anchor_id', 'nctId', 'document_id', 'anchor_type', 'anchor', 'positive', 'reference_answer', 'status', 'error']

Reference-answer types:
reference_answer
<class 'str'>    7866
Name: count, dtype: int64

Status:
status
success    7866
Name: count, dtype: int64

Unique documents:
2000

Rows per document — summary:
count    2000.000000
mean        3.933000
std         0.254054
min         2.000000
25%         4.000000
50%         4.000000
75%         4.000000
max         4.000000
dtype: float64

Sample:


,anchor_id,nctId,document_id,anchor_type,anchor,positive,reference_answer
0,0,NCT07640360,NCT07640360_1,macro_question,What is the main goal of this clinical trial r...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,The main goal of this clinical trial is to eva...
1,1,NCT07640360,NCT07640360_1,patient_profile_question,Could I qualify for this trial if I had a rece...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,"Yes, you could qualify for this trial if you h..."
2,2,NCT07640360,NCT07640360_1,operational_question,Is there an age limit for patients to particip...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,"Yes, there is an age limit for patients to par..."


In [23]:
SEED = 42
TEST_SIZE = 0.10

documents = ref_ans_df["document_id"].unique()

train_docs, test_docs = train_test_split(
    documents, test_size=TEST_SIZE, random_state=SEED
)

train_df = ref_ans_df[ref_ans_df["document_id"].isin(train_docs)].reset_index(drop=True)
test_df  = ref_ans_df[ref_ans_df["document_id"].isin(test_docs)].reset_index(drop=True)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Train docs: {len(train_docs)} | Test docs: {len(test_docs)}")
print(f"Document overlap: {len(set(train_docs) & set(test_docs))}")



Train: 7082 | Test: 784
Train docs: 1800 | Test docs: 200
Document overlap: 0


In [24]:
def generate_answers_batch(
    model,
    questions,
    retrieved_docs_batch,
    max_new_tokens=128,
):
    texts = []

    for question, retrieved_docs in zip(questions, retrieved_docs_batch):
        context = "\n\n".join(
            f"CONTEXT {i+1}:\n{doc.page_content}"
            for i, doc in enumerate(retrieved_docs)
        )

        messages = [
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"QUESTION:\n{question}\n\n{context}",
            },
        ]

        texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        )

    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    inputs = tokenizer(
        texts,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    tokenizer.padding_side = original_padding_side

    generated_tokens = outputs[:, inputs.input_ids.shape[1]:]

    return [
        tokenizer.decode(
            tokens,
            skip_special_tokens=True,
        ).strip()
        for tokens in generated_tokens
    ]


fixing pad token issue and adding same to configs

In [25]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

base_model.config.pad_token_id = tokenizer.pad_token_id
ft1_model.config.pad_token_id = tokenizer.pad_token_id

In [26]:
'''
# ============================================================
# BATCHED END-TO-END BASE vs FT RAG EVALUATION

eval_df = test_df[
    ["anchor_id", "nctId", "anchor", "positive", "reference_answer"]
].copy().reset_index(drop=True)

print(f"Evaluation examples: {len(eval_df):,}")
print(f"Unique trials: {eval_df['nctId'].nunique():,}")

assert len(eval_df) == 783

base_eval_results = []
ft_eval_results = []

BATCH_SIZE = 8

for start in tqdm(range(0, len(eval_df), BATCH_SIZE)):
    batch = eval_df.iloc[start:start + BATCH_SIZE]
    questions = batch["anchor"].tolist()

    base_docs = [
        langchain_retrieve(
            q,
            base_embed_model,
            base_faiss_index,
            corpus_df,
            top_k=4,
        )
        for q in questions
    ]

    ft_docs = [
        langchain_retrieve(
            q,
            ft_embed_model,
            ft_faiss_index,
            corpus_df,
            top_k=4,
        )
        for q in questions
    ]

    base_answers = generate_answers_batch(
        base_model,
        questions,
        base_docs,
    )

    ft_answers = generate_answers_batch(
        ft1_model,
        questions,
        ft_docs,
    )

    for row, answer, docs in zip(
        batch.itertuples(index=False),
        base_answers,
        base_docs,
    ):
        base_eval_results.append({
            "anchor_id": row.anchor_id,
            "nctId": row.nctId,
            "question": row.anchor,
            "reference_answer": row.reference_answer,
            "answer": answer,
            "retrieved_docs": docs,
        })

    for row, answer, docs in zip(
        batch.itertuples(index=False),
        ft_answers,
        ft_docs,
    ):
        ft_eval_results.append({
            "anchor_id": row.anchor_id,
            "nctId": row.nctId,
            "question": row.anchor,
            "reference_answer": row.reference_answer,
            "answer": answer,
            "retrieved_docs": docs,
        })

base_eval_df = pd.DataFrame(base_eval_results)
ft_eval_df = pd.DataFrame(ft_eval_results)

assert len(base_eval_df) == len(eval_df)
assert len(ft_eval_df) == len(eval_df)

print("Base:", len(base_eval_df))
print("FT + LoRA:", len(ft_eval_df))
'''

'\n# ============================================================\n# BATCHED END-TO-END BASE vs FT RAG EVALUATION\n\neval_df = test_df[\n    ["anchor_id", "nctId", "anchor", "positive", "reference_answer"]\n].copy().reset_index(drop=True)\n\nprint(f"Evaluation examples: {len(eval_df):,}")\nprint(f"Unique trials: {eval_df[\'nctId\'].nunique():,}")\n\nassert len(eval_df) == 783\n\nbase_eval_results = []\nft_eval_results = []\n\nBATCH_SIZE = 8\n\nfor start in tqdm(range(0, len(eval_df), BATCH_SIZE)):\n    batch = eval_df.iloc[start:start + BATCH_SIZE]\n    questions = batch["anchor"].tolist()\n\n    base_docs = [\n        langchain_retrieve(\n            q,\n            base_embed_model,\n            base_faiss_index,\n            corpus_df,\n            top_k=4,\n        )\n        for q in questions\n    ]\n\n    ft_docs = [\n        langchain_retrieve(\n            q,\n            ft_embed_model,\n            ft_faiss_index,\n            corpus_df,\n            top_k=4,\n        )\n   

In [27]:
# ============================================================
# BATCHED END-TO-END BASE vs FT RAG EVALUATION
# 400 diverse evaluation examples
# ============================================================

import gc
import torch
import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Build a diverse 400-example evaluation subset
#    Prefer maximum NCT coverage
# ------------------------------------------------------------

eval_source_df = test_df[
    ["anchor_id", "nctId", "anchor", "positive", "reference_answer"]
].copy().reset_index(drop=True)

# Shuffle deterministically, then take at most one example per NCT
# first, followed by additional examples if needed.
shuffled = eval_source_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# One random anchor per NCT first
diverse_first = shuffled.drop_duplicates(
    subset="nctId",
    keep="first"
)

# If fewer than 400 unique NCTs, fill remaining slots
remaining = shuffled[
    ~shuffled["anchor_id"].isin(diverse_first["anchor_id"])
]

n_needed = min(400, len(eval_source_df))#can choose any n_needed :{300, eval_source_df=784} in interest of time

if len(diverse_first) >= n_needed:
    eval_df = diverse_first.iloc[:n_needed].copy()
else:
    extra_needed = n_needed - len(diverse_first)

    eval_df = pd.concat(
        [
            diverse_first,
            remaining.iloc[:extra_needed]
        ],
        ignore_index=True
    )

eval_df = eval_df.reset_index(drop=True)

print(f"Evaluation examples: {len(eval_df):,}")
print(f"Unique trials: {eval_df['nctId'].nunique():,}")

assert len(eval_df) == min(400, len(eval_source_df))


# ------------------------------------------------------------
# 2. Settings
# ------------------------------------------------------------

BATCH_SIZE = 4
TOP_K = 4
MAX_NEW_TOKENS = 128

base_eval_results = []
ft_eval_results = []


# ------------------------------------------------------------
# 3. Batched evaluation
# ------------------------------------------------------------

for start in tqdm(
    range(0, len(eval_df), BATCH_SIZE),
    desc="End-to-end evaluation"
):

    batch = eval_df.iloc[start:start + BATCH_SIZE]
    questions = batch["anchor"].tolist()

    # --------------------------------------------------------
    # Retrieval
    # --------------------------------------------------------

    base_docs = [
        langchain_retrieve(
            q,
            base_embed_model,
            base_faiss_index,
            corpus_df,
            top_k=TOP_K,
        )
        for q in questions
    ]

    ft_docs = [
        langchain_retrieve(
            q,
            ft_embed_model,
            ft_faiss_index,
            corpus_df,
            top_k=TOP_K,
        )
        for q in questions
    ]

    # --------------------------------------------------------
    # BASE generation
    # --------------------------------------------------------

    base_answers = generate_answers_batch(
        base_model,
        questions,
        base_docs,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # Free temporary generation memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    # --------------------------------------------------------
    # FT + LoRA generation
    # --------------------------------------------------------

    ft_answers = generate_answers_batch(
        ft1_model,
        questions,
        ft_docs,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    for row, answer, docs in zip(
        batch.itertuples(index=False),
        base_answers,
        base_docs,
    ):
        base_eval_results.append({
            "anchor_id": row.anchor_id,
            "nctId": row.nctId,
            "question": row.anchor,
            "reference_answer": row.reference_answer,
            "answer": answer,
            "retrieved_docs": docs,
        })

    for row, answer, docs in zip(
        batch.itertuples(index=False),
        ft_answers,
        ft_docs,
    ):
        ft_eval_results.append({
            "anchor_id": row.anchor_id,
            "nctId": row.nctId,
            "question": row.anchor,
            "reference_answer": row.reference_answer,
            "answer": answer,
            "retrieved_docs": docs,
        })

    # --------------------------------------------------------
    # CHECKPOINT AFTER EVERY BATCH
    # --------------------------------------------------------

    base_checkpoint_df = pd.DataFrame(base_eval_results)
    ft_checkpoint_df = pd.DataFrame(ft_eval_results)

    base_checkpoint_df.to_pickle(
        "/kaggle/working/base_eval_checkpoint.pkl"
    )

    ft_checkpoint_df.to_pickle(
        "/kaggle/working/ft_eval_checkpoint.pkl"
    )


# ------------------------------------------------------------
# 4. Final DataFrames
# ------------------------------------------------------------

base_eval_df = pd.DataFrame(base_eval_results)
ft_eval_df = pd.DataFrame(ft_eval_results)

assert len(base_eval_df) == len(eval_df)
assert len(ft_eval_df) == len(eval_df)

print("============================================")
print("Evaluation completed")
print("============================================")
print("Base:", len(base_eval_df))
print("FT + LoRA:", len(ft_eval_df))
print("Unique NCTs:", eval_df["nctId"].nunique())

Evaluation examples: 400
Unique trials: 200


End-to-end evaluation:   0%|          | 0/100 [00:00<?, ?it/s]

Evaluation completed
Base: 400
FT + LoRA: 400
Unique NCTs: 200


In [28]:
import os
EVAL_DIR = "/kaggle/working/rag3_evaluation"#--->for kaggle notebook
#EVAL_DIR = "./rag3_evaluation"#for colab disc
os.makedirs(EVAL_DIR, exist_ok=True)

EVAL_PATH = os.path.join(EVAL_DIR, "base_vs_ft_RagPipeline_predictions.parquet")

eval_df.to_parquet(EVAL_PATH, index=False, engine='pyarrow')

print(f"Saved: {EVAL_PATH}")
print(f"Rows: {len(eval_df)}")
print(f"File size: {os.path.getsize(EVAL_PATH) / (1024**2):.2f} MB")

Saved: /kaggle/working/rag3_evaluation/base_vs_ft_RagPipeline_predictions.parquet
Rows: 400
File size: 0.24 MB


**BERT & rouge score and corresponding end to end summary**

In [29]:
# ============================================================
# PREPARE TEXTS FOR GENERATION METRICS
# ============================================================

base_predictions = base_eval_df["answer"].fillna("").astype(str).tolist()
ft_predictions = ft_eval_df["answer"].fillna("").astype(str).tolist()
references = eval_df["reference_answer"].fillna("").astype(str).tolist()

assert len(base_predictions) == len(references) == len(ft_predictions)

In [30]:
# ============================================================
# END-TO-END RAG: ROUGE-L + BERTSCORE
# ============================================================

from rouge_score import rouge_scorer
from bert_score import score as bertscore

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

base_rouge = [
    rouge.score(ref, pred)["rougeL"].fmeasure
    for ref, pred in zip(references, base_predictions)
]

ft_rouge = [
    rouge.score(ref, pred)["rougeL"].fmeasure
    for ref, pred in zip(references, ft_predictions)
]

_, _, base_bert = bertscore(
    base_predictions,
    references,
    lang="en",
    device="cuda",
)

_, _, ft_bert = bertscore(
    ft_predictions,
    references,
    lang="en",
    device="cuda",
)

print(f"Base ROUGE-L:    {sum(base_rouge) / len(base_rouge):.6f}")
print(f"FT ROUGE-L:      {sum(ft_rouge) / len(ft_rouge):.6f}")
print(f"Base BERTScore:  {base_bert.mean().item():.6f}")
print(f"FT BERTScore:    {ft_bert.mean().item():.6f}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base ROUGE-L:    0.479794
FT ROUGE-L:      0.512608
Base BERTScore:  0.906269
FT BERTScore:    0.911030


In [31]:
# ============================================================
# END-TO-END BASE vs FT METRIC SUMMARY
# ============================================================

metric_summary = pd.DataFrame({
    "Model": ["Base", "FT + LoRA"],
    "ROUGE-L": [
        sum(base_rouge) / len(base_rouge),
        sum(ft_rouge) / len(ft_rouge),
    ],
    "BERTScore-F1": [
        base_bert.mean().item(),
        ft_bert.mean().item(),
    ],
})

metric_summary["ROUGE-L Δ"] = metric_summary["ROUGE-L"] - metric_summary.loc[0, "ROUGE-L"]
metric_summary["BERTScore Δ"] = metric_summary["BERTScore-F1"] - metric_summary.loc[0, "BERTScore-F1"]

display(metric_summary)

,Model,ROUGE-L,BERTScore-F1,ROUGE-L Δ,BERTScore Δ
0,Base,0.479794,0.906269,0.000000,0.000000
1,FT + LoRA,0.512608,0.911030,0.032814,0.004761


**RAGAS ened to end evaluaition**

In [32]:
#!pip install langchain-google-vertexai
#!pip install -q requests==2.32.4
!pip install -q ragas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 9.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [31]:
#from langchain_google_vertexai import ChatVertexAI
import ragas
print(ragas.__version__)

ModuleNotFoundError: No module named 'ragas'